In [1]:
import tkinter as tk
from tkinter import filedialog
import pandas as pd
import os
import fnmatch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.widgets import LassoSelector
from scipy import signal
import numpy as np
%matplotlib qt

In [5]:
# Initialize tkinter root
root = tk.Tk()
root.withdraw()  # Hide the root window
root.attributes("-topmost", True)

# Define the initial directory
initial_directory = r"C:\Users\fossati.veronica\Desktop\Rewire_Project-master\Rewire_Project-master\Code\mediapipe\Data"  # Change this to your desired folder

#select one file for peak identification
file_path = filedialog.askopenfilename(
    initialdir=initial_directory, 
    filetypes=[("Filtered Csv Files", "*_mod*.csv")], 
    title="Select a Text File"
)

base_dir, file_name = os.path.split(file_path)
all_files = os.listdir(base_dir)
f1 = base_dir + "/"
f1 = f1 + fnmatch.filter(all_files, f"*{file_name.split('_')[1]}-startTime*.txt")[0]
f2 = base_dir + "/"
f2 = f2 + fnmatch.filter(all_files, f"*{file_name.split('_')[1]}_stimParams*.txt")[0]

data_kinematics = pd.read_csv(file_path)

with open(f1) as file:
    start_time = [line.rstrip() for line in file]
with open(f2) as file:
    stims = [line.rstrip() for line in file]
stim_events=[]
stim_type=[]
stim_params=[]

date_format = "%Y-%m-%d %H:%M:%S.%f"

seconds_delay = 0.5

for i_s,s in enumerate(stims):
    if 'min freq' in s:
        stim_events.append(pd.Timestamp(f'{os.path.split(f1)[1][:10]} '+stims[i_s-1])-pd.Timedelta(seconds=seconds_delay))  #MODIFY with different timepoint
        stim_type.append('freq')
        stim_params.append((int(stims[i_s+1].split(',')[4]),int(stims[i_s+1].split(',')[5]),int(stims[i_s+1].split(',')[6])))
    elif 'min amp' in s:
        stim_events.append(pd.Timestamp(f'{os.path.split(f1)[1][:10]} '+stims[i_s-1])-pd.Timedelta(seconds=seconds_delay))
        stim_type.append('amp')
        stim_params.append((int(stims[i_s+1].split(',')[2]),int(stims[i_s+1].split(',')[3]),int(stims[i_s+1].split(',')[4])))

In [6]:
data_kinematics

,Unnamed: 0,Thumb,Index,Middle,Ring,Pinky
0,2024-10-08 11:57:43.000000,167.255402,137.860307,127.736204,126.361600,134.450529
1,2024-10-08 11:57:43.032000,167.610274,138.262172,128.538739,127.452032,134.974853
2,2024-10-08 11:57:43.064067,167.959944,138.646405,129.320589,128.514100,135.478517
3,2024-10-08 11:57:43.096033,168.295557,139.007939,130.069652,129.526601,135.952122
4,2024-10-08 11:57:43.128067,168.609324,139.342383,130.775012,130.470491,136.387260
...,...,...,...,...,...,...
15776,2024-10-08 12:06:55.272333,170.500049,141.123608,131.953034,128.834352,138.302704
15777,2024-10-08 12:06:55.304333,170.429771,141.108983,131.934084,128.782481,138.283505
15778,2024-10-08 12:06:55.336400,170.362788,141.091099,131.911373,128.728862,138.262961
15779,2024-10-08 12:06:55.368400,170.300843,141.071170,131.886390,128.675340,138.241931


In [9]:
fig, axs = plt.subplots(len(data_kinematics.columns), 1, figsize=(10, 12), sharex=True)



#create stimulation pattern for plotting
time = []
for i in data_kinematics[data_kinematics.columns[0]].values:
    time.append(pd.Timestamp(i))
time_series=pd.Series(time)

stim_wave = []
timestamps=time_series[time_series < stim_events[0]]
timestamps = timestamps.tolist()
for ts in timestamps:
    stim_wave.append(0)

vertical_lines=[]
for i_stim,stim in enumerate(stim_params):
    n = 3   #to be changed properly
    n_trials=2
    current_value = 1  # Starting value for the step function
    steps=int((stim[1]-stim[0])/stim[2])+1

    if i_stim==0:
        timestamps = time_series[(time_series > stim_events[0]) & (time_series < stim_events[0]+pd.Timedelta(seconds=n*steps))]
    else:
        timestamps = time_series[(time_series > stim_events[i_stim]) & (time_series < stim_events[i_stim]+pd.Timedelta(seconds=n*steps))]
    
    timestamps = timestamps.tolist()
    next_change_time =  timestamps[0] + pd.Timedelta(seconds=n)
    vertical_lines.append(timestamps[0])
    
    for ts in timestamps:
        if ts >= next_change_time:
            vertical_lines.append(ts)
            current_value += 1  # Increment the step value at each interval
            next_change_time = ts + pd.Timedelta(seconds=n)  # Update to the next change time
        stim_wave.append(current_value)
    vertical_lines.append(timestamps[-1])

    if i_stim==len(stim_params)-1:
        timestamps=time_series[time_series > stim_events[i_stim]+pd.Timedelta(seconds=n*steps)]
    else:
        timestamps=time_series[(time_series > stim_events[i_stim]+pd.Timedelta(seconds=n*steps)) & (time_series < stim_events[i_stim+1])]
    timestamps=timestamps.tolist()
    for ts in timestamps:
        stim_wave.append(0)


#initialize GUI for peak identification: right button --> set threshold, left button and move --> move points, +/- keys --> add/remove points
def onselect(verts, i):

    threshold = verts[-1][1]   
    th_line[i].set_data(time_for_plot, [threshold]*len(time))
    
    # to find peaks based on the selected height
    peaks_temp, props = signal.find_peaks(x=-data_kinematics[data_kinematics.columns[i+1]].values, height=-threshold, distance=int(n/n_trials*24))
    peaks_temp = np.array(peaks_temp).astype(int)
    peaks_points[i].set_data(np.array(time_for_plot)[peaks_temp], data_kinematics[data_kinematics.columns[i+1]].values[peaks_temp])
    
    fig.canvas.draw_idle() 

def on_press(event):
    global press
    global index
    global closest_point_pressed
    
    if not any(event.inaxes == axis for axis in axs):
        return

    index = None
    for i, axis in enumerate(axs):
        if event.inaxes == axis:
            index = i-1
            break

    contains_point, attrd = peaks_points[index].contains(event)
    xdata_point = peaks_points[index].get_xdata()
    ydata_point = peaks_points[index].get_ydata()
    
    if not contains_point or index<0:
        return

    event_pixel_coords = axs[index+1].transData.transform((event.xdata, event.ydata))
    event_x_pixel, event_y_pixel = event_pixel_coords
    data_pixel_coords = axs[index+1].transData.transform(np.vstack([xdata_point, ydata_point]).T)
    x_pixel = data_pixel_coords[:, 0]
    y_pixel = data_pixel_coords[:, 1]
    distances = np.sqrt((x_pixel - event_x_pixel)**2 + (y_pixel - event_y_pixel)**2)
    if len(distances) > 0:
        closest_point_pressed = np.argmin(distances)
        press = (xdata_point[closest_point_pressed], ydata_point.astype('float64')[closest_point_pressed]), (event.xdata, event.ydata)
        
def on_motion(event):

    global press
    global index
    global closest_point_pressed

    if press is None or not any(event.inaxes == axis for axis in axs) or index<0:
        return
        
    (x0, y0), (xpress, ypress) = press
    dx = event.xdata - xpress
    dy = event.ydata - ypress

    xdata = peaks_points[index].get_xdata()
    ydata = peaks_points[index].get_ydata()
    idx = np.argmin(np.abs(lines[index].get_xdata()-(x0+dx)))
    x_new = lines[index].get_xdata()[idx]
    y_new = lines[index].get_ydata()[idx]
    xdata[closest_point_pressed] = x_new
    ydata[closest_point_pressed] = lines[index].get_ydata()[idx]
    peaks_points[index].set_data(xdata, ydata)
            
    fig.canvas.draw()

def on_release(event):
    
    global press
    global closest_point_pressed
    global index
    press = None
    closest_point = None

    sort_points(peaks_points[index])
    index=None
    fig.canvas.draw()

def sort_points(points):
    xdata = points.get_xdata()
    ydata = points.get_ydata()
    if len(xdata)>0:
        idx_sort = np.argsort(xdata)
        xdata=xdata[idx_sort]
        ydata=ydata[idx_sort]
        points.set_data(xdata, ydata)

def on_key_press(event):
    if not any(event.inaxes == axis for axis in axs):
        return

    index = None
    for i, axis in enumerate(axs):
        if event.inaxes == axis:
            index = i-1
            break
    
    if event.key == '+':
        xdata = lines[index].get_xdata()
        ydata = lines[index].get_ydata()
        event_pixel_coords = axs[index+1].transData.transform((event.xdata, event.ydata))
        event_x_pixel, event_y_pixel = event_pixel_coords
        data_pixel_coords = axs[index+1].transData.transform(np.vstack([xdata, ydata]).T)
        x_pixel = data_pixel_coords[:, 0]
        y_pixel = data_pixel_coords[:, 1]
        distances = np.sqrt((x_pixel - event_x_pixel)**2 + (y_pixel - event_y_pixel)**2)
        if len(distances) > 0:
            mindist = np.min(distances)
            if mindist > 5:
                return
            closest_point = np.argmin(distances)
            x_new = xdata[closest_point]
        peaks_points[index].set_data(np.append(peaks_points[index].get_xdata(), x_new), np.append(peaks_points[index].get_ydata(), ydata[closest_point]))
        sort_points(peaks_points[index])

    elif event.key == '-':
        xdata_points = peaks_points[index].get_xdata()
        ydata_points = peaks_points[index].get_ydata()
        event_pixel_coords = axs[index+1].transData.transform((event.xdata, event.ydata))
        event_x_pixel, event_y_pixel = event_pixel_coords
        data_pixel_coords = axs[index+1].transData.transform(np.vstack([xdata_points, ydata_points]).T)
        x_pixel = data_pixel_coords[:, 0]
        y_pixel = data_pixel_coords[:, 1]
        distances = np.sqrt((x_pixel - event_x_pixel)**2 + (y_pixel - event_y_pixel)**2)
        if len(distances) > 0:
            closest_point = np.argmin(distances)
            if distances[closest_point] < 6:  # Tolerance for click distance 
                xdata_points = np.delete(xdata_points, closest_point)
                ydata_points = np.delete(ydata_points, closest_point)
                peaks_points[index].set_data(xdata_points, ydata_points)
    fig.canvas.draw()

#initialize plot
time_for_plot=(mdates.date2num(time)-mdates.date2num(time)[0])
vertical_lines_for_plot=(mdates.date2num(vertical_lines)-mdates.date2num(time)[0])
peaks_points=[]
th_line=[]
lines=[]
for i, finger in enumerate(data_kinematics.columns):
    if i==0:
        axs[i].plot(time_for_plot,stim_wave)
        
    else:
        data_plot=data_kinematics[finger].values
        line,=axs[i].plot(time_for_plot,data_plot, label=finger, color='#9d0208') #
        lines.append(line)
        
        rec_curves_points, = axs[i].plot([],[], 'bo', markeredgecolor='black')
        peaks_points.append(rec_curves_points)

        ll, = axs[i].plot(time_for_plot, [data_kinematics[finger][0]]*len(time), 'r--', alpha=1, linewidth=0.5)
        th_line.append(ll)
        
        axs[i].set_title(finger)
        #axs[i].set_xlim((vertical_lines_for_plot[0],time_for_plot[-1]))

        for l in vertical_lines_for_plot:
            axs[i].axvline(l)

lasso_thumb = LassoSelector(axs[1], lambda verts: onselect(verts, i=0), button=3)
lasso_index = LassoSelector(axs[2], lambda verts: onselect(verts, i=1), button=3)
lasso_medium = LassoSelector(axs[3], lambda verts: onselect(verts, i=2), button=3)
lasso_ring = LassoSelector(axs[4], lambda verts: onselect(verts, i=3), button=3)
lasso_pinky = LassoSelector(axs[5], lambda verts: onselect(verts, i=4), button=3)

for i in range(5):
    peaks_points[i].set_picker(5)
    cidpress_hs = peaks_points[i].figure.canvas.mpl_connect(
        'button_press_event', on_press)
    cidrelease_hs = peaks_points[i].figure.canvas.mpl_connect(
        'button_release_event', on_release)
    cidmotion_hs = peaks_points[i].figure.canvas.mpl_connect(
        'motion_notify_event', on_motion)

fig.canvas.mpl_connect('key_press_event', on_key_press)

index=None
press=None
closest_point_pressed=None




figManager = plt.get_current_fig_manager()
figManager.window.showMaximized()

In [10]:
peaks_kinematics_abs=dict()
peaks_kinematics_common_ref=dict()
peaks_kinematics_single_ref=dict()
stim_vect=np.arange(stim_params[0][0], stim_params[0][1]+1, stim_params[0][2])
ref_common=[]

#compute peaks
for i, finger in enumerate(data_kinematics.columns[1:]):
    
    peaks_kinematics_abs[finger]=dict()  #absolute values of the angles reached during contraction
    peaks_kinematics_common_ref[finger]=dict()  #reference position is computed as mean of 2 secs before stim init
    peaks_kinematics_single_ref[finger]=dict() #single reference for each peak, as the resting position before each flexion
    
    peaks_x=peaks_points[i].get_xdata()
    peaks_y=peaks_points[i].get_ydata()
    peaks_y=np.delete(peaks_y, np.where(peaks_x<vertical_lines_for_plot[0]))
    peaks_x=np.delete(peaks_x, np.where(peaks_x<vertical_lines_for_plot[0]))
    peaks_y=np.delete(peaks_y, np.where(peaks_x>vertical_lines_for_plot[-1]))
    peaks_x=np.delete(peaks_x, np.where(peaks_x>vertical_lines_for_plot[-1]))

    single_ref=[]
    for el in peaks_x:
        idx_start=np.where(time_for_plot==el)[0]
        #print(idx_start)
        for aaa in np.arange(idx_start-1,0,-1):
            if data_kinematics[finger].values[aaa]-data_kinematics[finger].values[aaa-1]>=0:
                #print(aaa)
                single_ref.append(data_kinematics[finger].values[aaa])
                break
    ref_common.append(np.mean(data_kinematics[finger].values[np.where((time_for_plot>vertical_lines_for_plot[0]-2/(24*60*60)) & (time_for_plot<vertical_lines_for_plot[0]))]))

    for i_l, l in enumerate(vertical_lines_for_plot[:-1]):
        peaks_kinematics_abs[finger][stim_vect[i_l]] = peaks_y[np.where((peaks_x>l) & (peaks_x<vertical_lines_for_plot[i_l+1]))]
        peaks_kinematics_single_ref[finger][stim_vect[i_l]] = -peaks_y[np.where((peaks_x>l) & (peaks_x<vertical_lines_for_plot[i_l+1]))]+np.array(single_ref)[np.where((peaks_x>l) & (peaks_x<vertical_lines_for_plot[i_l+1]))]
        if peaks_kinematics_abs[finger][stim_vect[i_l]].shape[0]<n_trials:
            peaks_kinematics_single_ref[finger][stim_vect[i_l]] = np.concatenate((peaks_kinematics_single_ref[finger][stim_vect[i_l]],np.repeat(0,n_trials-peaks_kinematics_abs[finger][stim_vect[i_l]].shape[0])))
            peaks_kinematics_abs[finger][stim_vect[i_l]] = np.concatenate((peaks_kinematics_abs[finger][stim_vect[i_l]],np.repeat(0,n_trials-peaks_kinematics_abs[finger][stim_vect[i_l]].shape[0])))
            

        peaks_kinematics_common_ref[finger][stim_vect[i_l]]=np.empty((n_trials,))
        for kk in range(n_trials):
            if peaks_kinematics_abs[finger][stim_vect[i_l]][kk]==0:
                peaks_kinematics_common_ref[finger][stim_vect[i_l]][kk]=0.0
            else:
                peaks_kinematics_common_ref[finger][stim_vect[i_l]][kk]=-peaks_kinematics_abs[finger][stim_vect[i_l]][kk]+ref_common[i]
                if peaks_kinematics_common_ref[finger][stim_vect[i_l]][kk]<0:
                    peaks_kinematics_common_ref[finger][stim_vect[i_l]][kk]=0


C:\Users\fossati.veronica\AppData\Local\Temp\ipykernel_31500\2248522825.py:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  for aaa in np.arange(idx_start-1,0,-1):


In [11]:
peaks_kinematics_abs

{'Thumb': {300: array([0., 0.]),
  320: array([0., 0.]),
  340: array([127.34693838, 116.30387414]),
  360: array([111.89531354, 111.34859343]),
  380: array([107.29623699, 103.68260939]),
  400: array([100.0077916 , 102.80900616]),
  420: array([103.42984463, 101.26415191]),
  440: array([ 97.93309936, 101.22951631]),
  460: array([99.17668291, 98.8092237 ]),
  480: array([103.3138243 , 103.38383691]),
  500: array([96.4766634 , 98.51553297])},
 'Index': {300: array([0., 0.]),
  320: array([0., 0.]),
  340: array([0., 0.]),
  360: array([0., 0.]),
  380: array([0., 0.]),
  400: array([0., 0.]),
  420: array([0., 0.]),
  440: array([0., 0.]),
  460: array([0., 0.]),
  480: array([0., 0.]),
  500: array([0., 0.])},
 'Middle': {300: array([0., 0.]),
  320: array([0., 0.]),
  340: array([0., 0.]),
  360: array([0., 0.]),
  380: array([0., 0.]),
  400: array([0., 0.]),
  420: array([0., 0.]),
  440: array([0., 0.]),
  460: array([0., 0.]),
  480: array([0., 0.]),
  500: array([0., 0.])},
 

In [12]:
peaks_kinematics_common_ref

{'Thumb': {300: array([0., 0.]),
  320: array([0., 0.]),
  340: array([47.67469768, 58.71776192]),
  360: array([63.12632252, 63.67304263]),
  380: array([67.72539907, 71.33902667]),
  400: array([75.01384445, 72.2126299 ]),
  420: array([71.59179142, 73.75748415]),
  440: array([77.08853669, 73.79211974]),
  460: array([75.84495315, 76.21241236]),
  480: array([71.70781176, 71.63779914]),
  500: array([78.54497265, 76.50610309])},
 'Index': {300: array([0., 0.]),
  320: array([0., 0.]),
  340: array([0., 0.]),
  360: array([0., 0.]),
  380: array([0., 0.]),
  400: array([0., 0.]),
  420: array([0., 0.]),
  440: array([0., 0.]),
  460: array([0., 0.]),
  480: array([0., 0.]),
  500: array([0., 0.])},
 'Middle': {300: array([0., 0.]),
  320: array([0., 0.]),
  340: array([0., 0.]),
  360: array([0., 0.]),
  380: array([0., 0.]),
  400: array([0., 0.]),
  420: array([0., 0.]),
  440: array([0., 0.]),
  460: array([0., 0.]),
  480: array([0., 0.]),
  500: array([0., 0.])},
 'Ring': {300: 

In [13]:
peaks_kinematics_single_ref

{'Thumb': {300: array([0., 0.]),
  320: array([0., 0.]),
  340: array([51.90767201, 67.62804624]),
  360: array([73.451185  , 75.38652263]),
  380: array([79.25715695, 83.8109512 ]),
  400: array([88.83959558, 86.79128459]),
  420: array([85.20758649, 87.06512665]),
  440: array([88.65477792, 84.00921177]),
  460: array([84.91765354, 89.04169141]),
  480: array([81.58390801, 84.70552533]),
  500: array([87.83684761, 87.28857878])},
 'Index': {300: array([0., 0.]),
  320: array([0., 0.]),
  340: array([0., 0.]),
  360: array([0., 0.]),
  380: array([0., 0.]),
  400: array([0., 0.]),
  420: array([0., 0.]),
  440: array([0., 0.]),
  460: array([0., 0.]),
  480: array([0., 0.]),
  500: array([0., 0.])},
 'Middle': {300: array([0., 0.]),
  320: array([0., 0.]),
  340: array([0., 0.]),
  360: array([0., 0.]),
  380: array([0., 0.]),
  400: array([0., 0.]),
  420: array([0., 0.]),
  440: array([0., 0.]),
  460: array([0., 0.]),
  480: array([0., 0.]),
  500: array([0., 0.])},
 'Ring': {300: 

In [14]:
file_path_r=base_dir + "/" + file_name[11:15] + ".npy"
np.save(file_path_r, [peaks_kinematics_single_ref, peaks_kinematics_common_ref, peaks_kinematics_abs])